In [1]:
import pandas as pd
import numpy as np

EPS = 1e-9

# strict percentiles on training data (healthy)
WARN_PERCENTILE = 99.0
CRIT_PERCENTILE = 99.7

TRAIN_FRACTION = 0.8

# Top-k aggregation of per-feature deviations
TOP_K = 3


In [2]:
CSV_PATH = "cycle_data.csv"
df = pd.read_csv(CSV_PATH)

print("Rows, Cols:", df.shape)
df.head()


Rows, Cols: (1000, 16)


,timestamp,cycle_id,cycle_duration,belt_move_time,punch_down_time,punch_up_time,belt_forward_duration,belt_reverse_duration,punch_motor_down_duration,punch_motor_up_duration,cpu_temp_avg,punch_speed,belt_efficiency,machine_load,cycle_variance_10,cycle_trend_10
0,2025-12-12T12:44:45.074099,1,7.601437,3.133049,0.759846,0.759298,3.133049,2.949184,0.759846,0.759298,58.722098,1.316057,0.412165,1.519143,0.000000,0.000000
1,2025-12-12T12:44:52.474186,2,7.399120,2.975217,0.758720,0.737383,2.975217,2.927752,0.758720,0.737383,58.328358,1.318009,0.402104,1.496103,0.020466,-0.202317
2,2025-12-12T12:44:59.906448,3,7.431437,2.991865,0.742078,0.738353,2.991865,2.959081,0.742078,0.738353,58.810502,1.347568,0.402596,1.480431,0.023626,-0.170000
3,2025-12-12T12:45:07.326301,4,7.418940,2.979158,0.738496,0.760612,2.979158,2.940608,0.738496,0.760612,58.778032,1.354103,0.401561,1.499109,0.026183,-0.182498
4,2025-12-12T12:45:14.745775,5,7.418580,2.958776,0.762246,0.758214,2.958776,2.939282,0.762246,0.758214,58.440273,1.311912,0.398833,1.520460,0.027742,-0.182857


In [3]:
def build_ml_features_speed_invariant(df: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=df.index)

    # core time allocation ratios (invariant)
    X["belt_time_ratio"] = df["belt_move_time"] / (df["cycle_duration"] + EPS)
    X["load_ratio"] = df["machine_load"] / (df["cycle_duration"] + EPS)

    # symmetry ratios (invariant)
    X["punch_symmetry"] = df["punch_down_time"] / (df["punch_up_time"] + EPS)
    X["belt_symmetry"] = df["belt_forward_duration"] / (df["belt_reverse_duration"] + EPS)

    # punch-down share of punch motion (invariant)
    X["punch_down_share"] = df["punch_down_time"] / (df["punch_down_time"] + df["punch_up_time"] + EPS)

    # overhead share (invariant)
    X["overhead_ratio"] = 1.0 - (
        (df["belt_move_time"] + df["punch_down_time"] + df["punch_up_time"])
        / (df["cycle_duration"] + EPS)
    )

    # cleanup
    X = X.replace([np.inf, -np.inf], np.nan).dropna()
    return X


X_all = build_ml_features_speed_invariant(df)
print("X_all shape:", X_all.shape)
print("Features:", list(X_all.columns))
X_all.describe().T


X_all shape: (1000, 6)
Features: ['belt_time_ratio', 'load_ratio', 'punch_symmetry', 'belt_symmetry', 'punch_down_share', 'overhead_ratio']


,count,mean,std,min,25%,50%,75%,max
belt_time_ratio,1000.0,0.399029,0.001730,0.392913,0.397876,0.398999,0.400126,0.412165
load_ratio,1000.0,0.204557,0.002190,0.196403,0.203109,0.204662,0.206264,0.209553
punch_symmetry,1000.0,0.984066,0.015369,0.943225,0.970079,0.986790,0.998014,1.028936
belt_symmetry,1000.0,1.006643,0.006954,0.984820,1.003081,1.006365,1.011659,1.062344
punch_down_share,1000.0,0.495954,0.003908,0.485392,0.492406,0.496675,0.499503,0.507131
overhead_ratio,1000.0,0.396414,0.001783,0.387985,0.395178,0.396318,0.397596,0.402230


In [4]:
n = len(X_all)
split = int(n * TRAIN_FRACTION)

X_train = X_all.iloc[:split].copy()
X_val   = X_all.iloc[split:].copy()

print("Train:", X_train.shape, "Val:", X_val.shape)


Train: (800, 6) Val: (200, 6)


In [5]:
train_median = X_train.median()
train_mad = (X_train - train_median).abs().median().replace(0, EPS)

def robust_z_scores_row(x_row: pd.Series) -> pd.Series:
    return (x_row - train_median) / train_mad

def anomaly_score_from_z(z: pd.Series, top_k: int = TOP_K) -> float:
    absz = z.abs().sort_values(ascending=False)
    return float(absz.head(top_k).mean())

score_train = np.array([
    anomaly_score_from_z(robust_z_scores_row(row), TOP_K)
    for _, row in X_train.iterrows()
])

warning_thr = np.percentile(score_train, WARN_PERCENTILE)
critical_thr = np.percentile(score_train, CRIT_PERCENTILE)

print("Train score min/mean/max:", score_train.min(), score_train.mean(), score_train.max())
print("WARNING threshold:", warning_thr)
print("CRITICAL threshold:", critical_thr)


Train score min/mean/max: 0.1058843592736693 1.5863658555090503 9.92071458404847
WARNING threshold: 3.5865124873928282
CRITICAL threshold: 4.380225491092595


In [6]:
score_val = np.array([
    anomaly_score_from_z(robust_z_scores_row(row), TOP_K)
    for _, row in X_val.iterrows()
])

val_warn_rate = float(np.mean(score_val > warning_thr))
val_crit_rate = float(np.mean(score_val > critical_thr))

print("Validation WARNING rate:", val_warn_rate)
print("Validation CRITICAL rate:", val_crit_rate)
print("Val score min/mean/max:", score_val.min(), score_val.mean(), score_val.max())


Validation WARNING rate: 0.0
Validation CRITICAL rate: 0.0
Val score min/mean/max: 0.21841187173443521 1.4418690885409933 3.482740286979053


In [7]:
FEATURE_SUBSYSTEM = {
    "belt_time_ratio": "BELT",
    "belt_symmetry": "BELT",
    "load_ratio": "PUNCH",
    "punch_symmetry": "PUNCH",
    "punch_down_share": "PUNCH",
    "overhead_ratio": "SYSTEM",
}

# demo on first validation row
z_demo = robust_z_scores_row(X_val.iloc[0]).sort_values(key=lambda s: s.abs(), ascending=False)
print(z_demo.head(6))


belt_time_ratio     2.107911
punch_symmetry      1.640394
punch_down_share    1.625495
load_ratio         -1.553844
belt_symmetry       1.152163
overhead_ratio      0.185608
dtype: float64


In [8]:
def ml_predict_cycle(cycle_id: int, x_row: pd.Series) -> dict:
    z = robust_z_scores_row(x_row)
    score = anomaly_score_from_z(z, TOP_K)

    if score > critical_thr:
        status = "CRITICAL"
    elif score > warning_thr:
        status = "WARNING"
    else:
        status = "NORMAL"

    # confidence: margin-based (simple + explainable)
    if score <= warning_thr:
        confidence = float(max(0.0, 1.0 - (score / (warning_thr + EPS))))
    elif score >= critical_thr:
        confidence = 1.0
    else:
        confidence = float((score - warning_thr) / (critical_thr - warning_thr + EPS))

    # top 3 dominant features
    top = z.reindex(z.abs().sort_values(ascending=False).index).head(3)

    dominant = []
    subsystems = set()
    for name, zval in top.items():
        subsys = FEATURE_SUBSYSTEM.get(name, "SYSTEM")
        dominant.append({"name": name, "z_score": float(zval), "subsystem": subsys})
        subsystems.add(subsys)

    return {
        "layer": "ML_BEHAVIOR",
        "cycle_id": int(cycle_id),
        "anomaly_score": float(score),
        "thresholds": {"warning": float(warning_thr), "critical": float(critical_thr)},
        "status": status,
        "dominant_features": dominant,
        "affected_subsystems": sorted(list(subsystems)),
        "confidence": float(confidence),
    }

# demo: show 3 outputs
for i in range(3):
    idx = X_val.index[i]
    out = ml_predict_cycle(int(df.loc[idx, "cycle_id"]), X_all.loc[idx])
    print(out)


{'layer': 'ML_BEHAVIOR', 'cycle_id': 801, 'anomaly_score': 1.7912667325513294, 'thresholds': {'warning': 3.5865124873928282, 'critical': 4.380225491092595}, 'status': 'NORMAL', 'dominant_features': [{'name': 'belt_time_ratio', 'z_score': 2.1079111043823056, 'subsystem': 'BELT'}, {'name': 'punch_symmetry', 'z_score': 1.640394307984515, 'subsystem': 'PUNCH'}, {'name': 'punch_down_share', 'z_score': 1.6254947852871684, 'subsystem': 'PUNCH'}], 'affected_subsystems': ['BELT', 'PUNCH'], 'confidence': 0.5005547204008137}
{'layer': 'ML_BEHAVIOR', 'cycle_id': 802, 'anomaly_score': 1.420824961485259, 'thresholds': {'warning': 3.5865124873928282, 'critical': 4.380225491092595}, 'status': 'NORMAL', 'dominant_features': [{'name': 'punch_down_share', 'z_score': -1.9240089430433056, 'subsystem': 'PUNCH'}, {'name': 'punch_symmetry', 'z_score': -1.8947987204486185, 'subsystem': 'PUNCH'}, {'name': 'belt_time_ratio', 'z_score': 0.4436672209638534, 'subsystem': 'BELT'}], 'affected_subsystems': ['BELT', 'P

In [9]:
def simulate_speed_change(df_one: pd.Series, scale: float) -> pd.Series:
    """
    Simulate global speed scaling:
    scale < 1 -> faster (times decrease)
    scale > 1 -> slower (times increase)
    """
    s = df_one.copy()

    # scale all time-like fields consistently
    time_cols = [
        "cycle_duration", "belt_move_time", "punch_down_time", "punch_up_time",
        "belt_forward_duration", "belt_reverse_duration",
        "punch_motor_down_duration", "punch_motor_up_duration",
        "machine_load"
    ]
    for col in time_cols:
        if col in s:
            s[col] = s[col] * scale

    # punch_speed = 1/punch_down_time -> scales inversely
    if "punch_speed" in s:
        s["punch_speed"] = s["punch_speed"] / scale

    # belt_efficiency ideally remains belt_move/cycle (invariant)
    if "belt_efficiency" in s:
        s["belt_efficiency"] = s["belt_move_time"] / (s["cycle_duration"] + EPS)

    return s


row = df.iloc[100].copy()

for scale in [0.9, 1.0, 1.1]:
    sim_row = simulate_speed_change(row, scale)
    sim_X = build_ml_features_speed_invariant(pd.DataFrame([sim_row])).iloc[0]
    out = ml_predict_cycle(int(row["cycle_id"]), sim_X)
    print("scale:", scale, "status:", out["status"], "score:", out["anomaly_score"])


scale: 0.9 status: NORMAL score: 0.9919683986938373
scale: 1.0 status: NORMAL score: 0.9919683928847439
scale: 1.1 status: NORMAL score: 0.9919683881318764


In [10]:
def inject_punch_wear(df_one: pd.Series, down_multiplier: float = 1.3) -> pd.Series:
    """
    Simulate wear: punch_down_time increases, others unchanged.
    This should change punch_symmetry and punch_down_share -> anomaly.
    """
    s = df_one.copy()
    s["punch_down_time"] *= down_multiplier
    s["punch_motor_down_duration"] *= down_multiplier

    # machine_load changes too
    s["machine_load"] = s["punch_down_time"] + s["punch_up_time"]

    # cycle_duration likely increases (not always exactly, but reasonable)
    s["cycle_duration"] = (
        s["belt_move_time"] + s["punch_down_time"] + s["punch_up_time"]
        + (s["cycle_duration"] - (s["belt_move_time"] + (df_one["punch_down_time"]) + s["punch_up_time"]))
    )

    # recompute punch_speed
    s["punch_speed"] = 1.0 / (s["punch_down_time"] + EPS)

    # belt_efficiency update
    s["belt_efficiency"] = s["belt_move_time"] / (s["cycle_duration"] + EPS)
    return s


base = df.iloc[100].copy()
wear = inject_punch_wear(base, down_multiplier=1.3)

X_wear = build_ml_features_speed_invariant(pd.DataFrame([wear])).iloc[0]
out_wear = ml_predict_cycle(int(base["cycle_id"]), X_wear)

print(out_wear)


{'layer': 'ML_BEHAVIOR', 'cycle_id': 101, 'anomaly_score': 18.31621276458052, 'thresholds': {'warning': 3.5865124873928282, 'critical': 4.380225491092595}, 'status': 'CRITICAL', 'dominant_features': [{'name': 'punch_symmetry', 'z_score': 22.225227526733214, 'subsystem': 'PUNCH'}, {'name': 'punch_down_share', 'z_score': 19.310469415547825, 'subsystem': 'PUNCH'}, {'name': 'load_ratio', 'z_score': 13.412941351460525, 'subsystem': 'PUNCH'}], 'affected_subsystems': ['PUNCH'], 'confidence': 1.0}


In [11]:
ml_model = {
    "model_type": "robust_zscore_topk_speed_invariant",
    "feature_names": list(X_all.columns),
    "train_median": train_median.to_dict(),
    "train_mad": train_mad.to_dict(),
    "threshold_warning": float(warning_thr),
    "threshold_critical": float(critical_thr),
    "top_k": int(TOP_K),
    "feature_subsystem": FEATURE_SUBSYSTEM,
}
ml_model


{'model_type': 'robust_zscore_topk_speed_invariant',
 'feature_names': ['belt_time_ratio',
  'load_ratio',
  'punch_symmetry',
  'belt_symmetry',
  'punch_down_share',
  'overhead_ratio'],
 'train_median': {'belt_time_ratio': 0.39898397654520645,
  'load_ratio': 0.20454202784055586,
  'punch_symmetry': 0.9863260893021688,
  'belt_symmetry': 1.006072115598187,
  'punch_down_share': 0.4965579890630624,
  'overhead_ratio': 0.39641470786445016},
 'train_mad': {'belt_time_ratio': 0.001121829752416148,
  'load_ratio': 0.0016309148765862869,
  'punch_symmetry': 0.013709816923457185,
  'belt_symmetry': 0.005036524058422476,
  'punch_down_share': 0.00346739836776494,
  'overhead_ratio': 0.001232476185509035},
 'threshold_warning': 3.5865124873928282,
 'threshold_critical': 4.380225491092595,
 'top_k': 3,
 'feature_subsystem': {'belt_time_ratio': 'BELT',
  'belt_symmetry': 'BELT',
  'load_ratio': 'PUNCH',
  'punch_symmetry': 'PUNCH',
  'punch_down_share': 'PUNCH',
  'overhead_ratio': 'SYSTEM'}}

In [12]:
def predict_df(df_features: pd.DataFrame, cycle_ids: pd.Series) -> pd.DataFrame:
    """
    Run ml_predict_cycle on each row and return a dataframe of results.
    """
    rows = []
    for idx, x_row in df_features.iterrows():
        cid = int(cycle_ids.loc[idx])
        out = ml_predict_cycle(cid, x_row)
        rows.append(out)
    return pd.DataFrame(rows)

def status_counts(results_df: pd.DataFrame) -> dict:
    return results_df["status"].value_counts().to_dict()

def rate(results_df: pd.DataFrame, status: str) -> float:
    return float(np.mean(results_df["status"] == status))

def summarize_results(name: str, results_df: pd.DataFrame):
    print("\n====", name, "====")
    print("Counts:", status_counts(results_df))
    print("WARNING rate:", rate(results_df, "WARNING"))
    print("CRITICAL rate:", rate(results_df, "CRITICAL"))
    print("Mean score:", float(results_df["anomaly_score"].mean()))
    print("Max score:", float(results_df["anomaly_score"].max()))


In [14]:
healthy_results = predict_df(X_all, df.loc[X_all.index, "cycle_id"])
summarize_results("HEALTHY DATA (PWM=80 training file)", healthy_results)



==== HEALTHY DATA (PWM=80 training file) ====
Counts: {'NORMAL': 992, 'WARNING': 5, 'CRITICAL': 3}
WARNING rate: 0.005
CRITICAL rate: 0.003
Mean score: 1.557466502115439
Max score: 9.92071458404847


In [15]:
def simulate_speed_df(df_in: pd.DataFrame, scale: float) -> pd.DataFrame:
    sim_rows = []
    for _, row in df_in.iterrows():
        sim_rows.append(simulate_speed_change(row, scale))
    return pd.DataFrame(sim_rows)

# choose a sample of cycles (e.g., 200 random-ish, but keep it deterministic by slicing)
sample_idx = df.index[:200]
df_sample = df.loc[sample_idx].copy()

scales = [0.85, 0.9, 1.0, 1.1, 1.15]
for sc in scales:
    df_sim = simulate_speed_df(df_sample, sc)
    X_sim = build_ml_features_speed_invariant(df_sim)
    res = predict_df(X_sim, df_sim.loc[X_sim.index, "cycle_id"])
    summarize_results(f"SPEED SCALE TEST scale={sc}", res)



==== SPEED SCALE TEST scale=0.85 ====
Counts: {'NORMAL': 195, 'CRITICAL': 3, 'WARNING': 2}
WARNING rate: 0.01
CRITICAL rate: 0.015
Mean score: 1.9184091781228698
Max score: 9.920714573155527

==== SPEED SCALE TEST scale=0.9 ====
Counts: {'NORMAL': 195, 'CRITICAL': 3, 'WARNING': 2}
WARNING rate: 0.01
CRITICAL rate: 0.015
Mean score: 1.9184091765569042
Max score: 9.920714577189896

==== SPEED SCALE TEST scale=1.0 ====
Counts: {'NORMAL': 195, 'CRITICAL': 3, 'WARNING': 2}
WARNING rate: 0.01
CRITICAL rate: 0.015
Mean score: 1.9184091738947706
Max score: 9.92071458404847

==== SPEED SCALE TEST scale=1.1 ====
Counts: {'NORMAL': 195, 'CRITICAL': 3, 'WARNING': 2}
WARNING rate: 0.01
CRITICAL rate: 0.015
Mean score: 1.9184091717166591
Max score: 9.920714589659942

==== SPEED SCALE TEST scale=1.15 ====
Counts: {'NORMAL': 195, 'CRITICAL': 3, 'WARNING': 2}
WARNING rate: 0.01
CRITICAL rate: 0.015
Mean score: 1.9184091707696522
Max score: 9.920714592099847


In [21]:
def inject_punch_down_wear(s: pd.Series, mult: float) -> pd.Series:
    s = s.copy()
    s["punch_down_time"] *= mult
    s["punch_motor_down_duration"] *= mult
    s["machine_load"] = s["punch_down_time"] + s["punch_up_time"]
    s["cycle_duration"] = s["belt_move_time"] + s["punch_down_time"] + s["punch_up_time"] + (s["cycle_duration"] - (s["belt_move_time"] + (s["punch_down_time"]/mult) + s["punch_up_time"]))
    s["punch_speed"] = 1.0 / (s["punch_down_time"] + EPS)
    s["belt_efficiency"] = s["belt_move_time"] / (s["cycle_duration"] + EPS)
    return s

def inject_punch_up_wear(s: pd.Series, mult: float) -> pd.Series:
    s = s.copy()
    s["punch_up_time"] *= mult
    s["punch_motor_up_duration"] *= mult
    s["machine_load"] = s["punch_down_time"] + s["punch_up_time"]
    s["cycle_duration"] = s["belt_move_time"] + s["punch_down_time"] + s["punch_up_time"] + (s["cycle_duration"] - (s["belt_move_time"] + s["punch_down_time"] + (s["punch_up_time"]/mult)))
    s["punch_speed"] = 1.0 / (s["punch_down_time"] + EPS)
    s["belt_efficiency"] = s["belt_move_time"] / (s["cycle_duration"] + EPS)
    return s

def inject_belt_forward_slow(s: pd.Series, mult: float) -> pd.Series:
    s = s.copy()
    s["belt_move_time"] *= mult
    s["belt_forward_duration"] *= mult
    # cycle_duration increases (assume overhead constant)
    s["cycle_duration"] = s["belt_move_time"] + s["punch_down_time"] + s["punch_up_time"] + (s["cycle_duration"] - ((s["belt_move_time"]/mult) + s["punch_down_time"] + s["punch_up_time"]))
    s["belt_efficiency"] = s["belt_move_time"] / (s["cycle_duration"] + EPS)
    return s

def inject_overhead_increase(s: pd.Series, extra_seconds: float) -> pd.Series:
    s = s.copy()
    s["cycle_duration"] += extra_seconds
    s["belt_efficiency"] = s["belt_move_time"] / (s["cycle_duration"] + EPS)
    return s


def apply_injection(df_in: pd.DataFrame, fn, *args) -> pd.DataFrame:
    out = []
    for _, row in df_in.iterrows():
        out.append(fn(row, *args))
    return pd.DataFrame(out)


In [18]:
# use same sample set
df_base = df_sample.copy()

tests = [
    ("PUNCH_DOWN_WEAR", inject_punch_down_wear, [1.05, 1.10, 1.20, 1.30]),
    ("PUNCH_UP_WEAR", inject_punch_up_wear, [1.05, 1.10, 1.20, 1.30]),
    ("BELT_FORWARD_SLOW", inject_belt_forward_slow, [1.05, 1.10, 1.20]),
    ("OVERHEAD_INCREASE", inject_overhead_increase, [0.05, 0.10, 0.20, 0.50]),
]

for name, fn, levels in tests:
    for lvl in levels:
        df_inj = apply_injection(df_base, fn, lvl)
        X_inj = build_ml_features_speed_invariant(df_inj)
        res = predict_df(X_inj, df_inj.loc[X_inj.index, "cycle_id"])
        summarize_results(f"{name} level={lvl}", res)



==== PUNCH_DOWN_WEAR level=1.05 ====
Counts: {'NORMAL': 146, 'WARNING': 45, 'CRITICAL': 9}
WARNING rate: 0.225
CRITICAL rate: 0.045
Mean score: 3.0223399587974895
Max score: 9.83351290948168

==== PUNCH_DOWN_WEAR level=1.1 ====
Counts: {'CRITICAL': 186, 'WARNING': 14}
WARNING rate: 0.07
CRITICAL rate: 0.93
Mean score: 5.853282118719612
Max score: 9.825757259023476

==== PUNCH_DOWN_WEAR level=1.2 ====
Counts: {'CRITICAL': 200}
WARNING rate: 0.0
CRITICAL rate: 1.0
Mean score: 11.788709136864268
Max score: 14.584421240463678

==== PUNCH_DOWN_WEAR level=1.3 ====
Counts: {'CRITICAL': 200}
WARNING rate: 0.0
CRITICAL rate: 1.0
Mean score: 17.62085080298125
Max score: 20.568754194659974

==== PUNCH_UP_WEAR level=1.05 ====
Counts: {'NORMAL': 113, 'WARNING': 66, 'CRITICAL': 21}
WARNING rate: 0.33
CRITICAL rate: 0.105
Mean score: 3.319062447714949
Max score: 9.833575500901848

==== PUNCH_UP_WEAR level=1.1 ====
Counts: {'CRITICAL': 194, 'WARNING': 5, 'NORMAL': 1}
WARNING rate: 0.025
CRITICAL rate

In [19]:
# Pick one scenario to inspect
df_inj = apply_injection(df_base.iloc[:5], inject_punch_down_wear, 1.20)
X_inj = build_ml_features_speed_invariant(df_inj)

for i in range(len(X_inj)):
    out = ml_predict_cycle(int(df_inj.iloc[i]["cycle_id"]), X_inj.iloc[i])
    print(out["status"], out["anomaly_score"], out["dominant_features"])


CRITICAL 14.270579862461846 [{'name': 'punch_symmetry', 'z_score': 15.648653103992945, 'subsystem': 'PUNCH'}, {'name': 'punch_down_share', 'z_score': 14.153396800823048, 'subsystem': 'PUNCH'}, {'name': 'overhead_ratio', 'z_score': -13.009689682569542, 'subsystem': 'SYSTEM'}]
CRITICAL 14.217191040244055 [{'name': 'punch_symmetry', 'z_score': 18.118235669944188, 'subsystem': 'PUNCH'}, {'name': 'punch_down_share', 'z_score': 16.138732547255895, 'subsystem': 'PUNCH'}, {'name': 'load_ratio', 'z_score': 8.39460490353208, 'subsystem': 'PUNCH'}]
CRITICAL 12.278098307110445 [{'name': 'punch_symmetry', 'z_score': 16.02697139514937, 'subsystem': 'PUNCH'}, {'name': 'punch_down_share', 'z_score': 14.461485432357117, 'subsystem': 'PUNCH'}, {'name': 'load_ratio', 'z_score': 6.345838093824842, 'subsystem': 'PUNCH'}]
CRITICAL 11.02035473130004 [{'name': 'punch_symmetry', 'z_score': 13.040444248997655, 'subsystem': 'PUNCH'}, {'name': 'punch_down_share', 'z_score': 11.989198636263689, 'subsystem': 'PUNCH

In [22]:
import joblib

MODEL_PATH = "ml_behavior_model.joblib"

joblib.dump(ml_model, MODEL_PATH)
print("Model saved to:", MODEL_PATH)


Model saved to: ml_behavior_model.joblib
